In [1]:
# ============================================================
# NB08_Paper_Trader
# Paper-trading simulation only; no broker connection.
#
# Reads:
#   gold_stock_candidates
#   silver_price_features
#   gold_paper_portfolio
#   gold_paper_cash
#
# Writes:
#   gold_paper_portfolio
#   gold_paper_cash
#   gold_paper_ledger
# ============================================================

import uuid
import numpy as np
import pandas as pd

from datetime import datetime, date
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    LongType, DateType
)

RUN_ID = str(uuid.uuid4())
TODAY = datetime.utcnow().date()

# Simulation settings
MAX_POSITION_SIZE = 50_000.0
MIN_CASH_TO_BUY = 10_000.0
STOP_LOSS_PCT = 0.07
MAX_PRICE_AGE_DAYS = 10

# Preserve the existing strategy rule. Set to False for a longer-term
# simulation that only exits on stop-loss.
SELL_WHEN_DROPPED_FROM_CANDIDATES = True

print(f"Run ID: {RUN_ID}")
print(f"Paper-trading date: {TODAY}")


# ------------------------------------------------------------
# Schemas
# ------------------------------------------------------------

PORTFOLIO_SCHEMA = StructType([
    StructField("symbol", StringType(), False),
    StructField("buy_date", DateType(), True),
    StructField("buy_price", DoubleType(), False),
    StructField("quantity", LongType(), False),
    StructField("invested_amount", DoubleType(), False),
])

CASH_SCHEMA = StructType([
    StructField("account", StringType(), False),
    StructField("cash_balance", DoubleType(), False),
    StructField("last_updated", DateType(), True),
])

LEDGER_SCHEMA = StructType([
    StructField("symbol", StringType(), True),
    StructField("buy_date", DateType(), True),
    StructField("sell_date", DateType(), True),
    StructField("buy_price", DoubleType(), True),
    StructField("sell_price", DoubleType(), True),
    StructField("quantity", LongType(), True),
    StructField("realized_pnl", DoubleType(), True),
    StructField("pnl_pct", DoubleType(), True),
    StructField("reason", StringType(), True),
])


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def finite_float(value):
    """Return a finite float, or None for null/NaN/infinite/non-numeric values."""
    try:
        if value is None or pd.isna(value):
            return None
        result = float(value)
        return result if np.isfinite(result) else None
    except (TypeError, ValueError):
        return None


def to_date(value):
    """Convert a date-like value to datetime.date; preserve null as None."""
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
        return pd.to_datetime(value).date()
    except (TypeError, ValueError):
        raise ValueError(f"Invalid date value: {value!r}")


def normalize_symbol(value):
    if value is None or pd.isna(value):
        return None
    text = str(value).strip().upper()
    return text if text not in {"", "NAN", "NONE", "NULL"} else None


# ------------------------------------------------------------
# 1. Load and validate the simulated account
# ------------------------------------------------------------

portfolio = spark.read.table("gold_paper_portfolio").toPandas()
cash_table = spark.read.table("gold_paper_cash").toPandas()

required_portfolio_columns = {
    "symbol", "buy_date", "buy_price", "quantity"
}
missing_columns = required_portfolio_columns - set(portfolio.columns)

if missing_columns:
    raise ValueError(
        "gold_paper_portfolio is missing required columns: "
        f"{sorted(missing_columns)}. Check the one-time initialization table."
    )

if cash_table.empty or "cash_balance" not in cash_table.columns:
    raise ValueError(
        "gold_paper_cash has no usable cash_balance. "
        "Run the one-time paper-account initialization notebook."
    )

cash_values = pd.to_numeric(cash_table["cash_balance"], errors="coerce")
cash_values = cash_values[cash_values.notna() & np.isfinite(cash_values)]

if cash_values.empty:
    raise ValueError("gold_paper_cash contains no finite cash_balance.")

cash_balance = float(cash_values.iloc[-1])

if cash_balance < 0:
    raise ValueError(f"Paper cash cannot be negative: {cash_balance}")

# Clean and validate existing open positions.
if not portfolio.empty:
    portfolio["symbol"] = portfolio["symbol"].apply(normalize_symbol)
    portfolio = portfolio[portfolio["symbol"].notna()].copy()

    portfolio["buy_price"] = pd.to_numeric(
        portfolio["buy_price"], errors="coerce"
    )
    portfolio["quantity"] = pd.to_numeric(
        portfolio["quantity"], errors="coerce"
    )

    invalid_position = (
        portfolio["buy_price"].isna()
        | ~np.isfinite(portfolio["buy_price"])
        | (portfolio["buy_price"] <= 0)
        | portfolio["quantity"].isna()
        | ~np.isfinite(portfolio["quantity"])
        | (portfolio["quantity"] <= 0)
        | (portfolio["quantity"] % 1 != 0)
    )

    if invalid_position.any():
        print("Invalid open-position rows:")
        display(portfolio.loc[invalid_position])
        raise ValueError(
            "The paper portfolio has an invalid buy_price or quantity. "
            "Fix those rows before running the simulation."
        )

    portfolio["quantity"] = portfolio["quantity"].astype("int64")

print(f"Starting simulated cash: ₹{cash_balance:,.2f}")
print(f"Existing simulated positions: {len(portfolio)}")


# ------------------------------------------------------------
# 2. Load latest valid price per symbol
# ------------------------------------------------------------

price_source = spark.read.table("silver_price_features")

price_clean = (
    price_source
    .withColumn("trade_date", F.to_date("trade_date"))
    .withColumn("_close", F.col("close").cast("double"))
    .filter(F.col("symbol").isNotNull())
    .filter(F.col("trade_date").isNotNull())
    .filter(F.col("_close").isNotNull())
    .filter(~F.isnan(F.col("_close")))
    .filter(F.col("_close") > 0)
)

w_latest = Window.partitionBy("symbol").orderBy(F.col("trade_date").desc())

latest_prices = (
    price_clean
    .withColumn("rn", F.row_number().over(w_latest))
    .filter(F.col("rn") == 1)
    .select(
        F.upper(F.trim(F.col("symbol"))).alias("symbol"),
        "trade_date",
        F.col("_close").alias("close")
    )
    .toPandas()
)

price_by_symbol = {}

for row in latest_prices.itertuples(index=False):
    symbol = normalize_symbol(row.symbol)
    price = finite_float(row.close)
    price_date = to_date(row.trade_date)

    if not symbol or price is None or price <= 0 or price_date is None:
        continue

    age_days = (TODAY - price_date).days

    # A stale quote is not used for a simulated buy or sell.
    if age_days > MAX_PRICE_AGE_DAYS:
        continue

    price_by_symbol[symbol] = {
        "close": price,
        "trade_date": price_date
    }

print(f"Symbols with a recent valid price: {len(price_by_symbol)}")


# ------------------------------------------------------------
# 3. Load and validate current candidates
# ------------------------------------------------------------

candidates = spark.read.table("gold_stock_candidates").toPandas()

if not candidates.empty:
    if "symbol" not in candidates.columns:
        raise ValueError("gold_stock_candidates has no symbol column.")

    candidates["symbol"] = candidates["symbol"].apply(normalize_symbol)
    candidates = candidates[candidates["symbol"].notna()].copy()

    if "candidate_rank" in candidates.columns:
        candidates["candidate_rank"] = pd.to_numeric(
            candidates["candidate_rank"], errors="coerce"
        )
        candidates = candidates.sort_values(
            "candidate_rank", na_position="last"
        )
    else:
        print("Warning: candidate_rank is missing; using table order.")

    # Remove duplicate symbols; retain the highest-ranked occurrence.
    candidates = candidates.drop_duplicates(subset=["symbol"], keep="first")

candidate_symbols = (
    set(candidates["symbol"].tolist())
    if not candidates.empty else set()
)

print(f"Candidate symbols: {len(candidate_symbols)}")

# Candidate prices are taken from the latest valid per-symbol price table.
buy_candidates = []
skipped_candidate_prices = []

for _, row in candidates.iterrows():
    symbol = row["symbol"]
    price_info = price_by_symbol.get(symbol)

    if price_info is None:
        skipped_candidate_prices.append(symbol)
        continue

    buy_candidates.append({
        "symbol": symbol,
        "price": price_info["close"],
        "price_date": price_info["trade_date"],
        "candidate_rank": row.get("candidate_rank")
    })

if skipped_candidate_prices:
    print(
        "Skipped candidates without a recent valid price:",
        skipped_candidate_prices
    )


# ------------------------------------------------------------
# 4. Evaluate current positions and simulate sells
# ------------------------------------------------------------

active_positions = []
ledger_rows = []
sold_this_run = set()

for idx, row in portfolio.iterrows():
    symbol = normalize_symbol(row["symbol"])
    buy_price = finite_float(row["buy_price"])
    quantity_value = finite_float(row["quantity"])

    if not symbol or buy_price is None or buy_price <= 0:
        raise ValueError(f"Invalid holding at row {idx}: {row.to_dict()}")

    if (
        quantity_value is None
        or quantity_value <= 0
        or quantity_value % 1 != 0
    ):
        raise ValueError(f"Invalid quantity for {symbol}: {row['quantity']}")

    quantity = int(quantity_value)
    buy_date = to_date(row["buy_date"])

    invested_amount = finite_float(row.get("invested_amount"))
    if invested_amount is None or invested_amount <= 0:
        invested_amount = buy_price * quantity

    price_info = price_by_symbol.get(symbol)

    # No recent quote: keep position unchanged. Never invent a price.
    if price_info is None:
        print(f"⚠️ Keeping {symbol}: no recent valid price available.")
        active_positions.append({
            "symbol": symbol,
            "buy_date": buy_date,
            "buy_price": buy_price,
            "quantity": quantity,
            "invested_amount": invested_amount
        })
        continue

    current_price = price_info["close"]
    stop_price = buy_price * (1.0 - STOP_LOSS_PCT)

    sell = False
    reason = None

    if current_price <= stop_price:
        sell = True
        reason = "STOP_LOSS_TRIGGERED"
    elif (
        SELL_WHEN_DROPPED_FROM_CANDIDATES
        and len(candidate_symbols) > 0
        and symbol not in candidate_symbols
    ):
        sell = True
        reason = "DROPPED_FROM_CANDIDATES"

    if sell:
        proceeds = current_price * quantity
        realized_pnl = proceeds - (buy_price * quantity)
        pnl_pct = (current_price / buy_price - 1.0) * 100.0

        cash_balance += proceeds
        sold_this_run.add(symbol)

        ledger_rows.append({
            "symbol": symbol,
            "buy_date": buy_date,
            "sell_date": TODAY,
            "buy_price": float(buy_price),
            "sell_price": float(current_price),
            "quantity": int(quantity),
            "realized_pnl": float(realized_pnl),
            "pnl_pct": float(pnl_pct),
            "reason": reason
        })

        print(
            f"Paper SELL: {symbol} at ₹{current_price:,.2f}; "
            f"reason={reason}; P&L=₹{realized_pnl:,.2f}"
        )
    else:
        active_positions.append({
            "symbol": symbol,
            "buy_date": buy_date,
            "buy_price": buy_price,
            "quantity": quantity,
            "invested_amount": invested_amount
        })


# ------------------------------------------------------------
# 5. Simulate buys
# ------------------------------------------------------------

held_symbols = {p["symbol"] for p in active_positions}
blocked_symbols = held_symbols | sold_this_run

for candidate in buy_candidates:
    if cash_balance < MIN_CASH_TO_BUY:
        print("Not enough simulated cash for another purchase.")
        break

    symbol = candidate["symbol"]

    if symbol in blocked_symbols:
        continue

    price = finite_float(candidate["price"])

    # Critical guard: never convert NaN or an invalid price to integer quantity.
    if price is None or price <= 0:
        print(f"⚠️ Skipped {symbol}: invalid price {candidate['price']!r}")
        continue

    invest_amount = min(float(MAX_POSITION_SIZE), float(cash_balance))
    quantity_float = np.floor(invest_amount / price)

    if not np.isfinite(quantity_float) or quantity_float < 1:
        print(f"⚠️ Skipped {symbol}: allocation cannot buy one share.")
        continue

    quantity = int(quantity_float)
    actual_cost = float(quantity * price)

    if actual_cost <= 0 or actual_cost > cash_balance:
        print(f"⚠️ Skipped {symbol}: invalid calculated cost {actual_cost}.")
        continue

    cash_balance -= actual_cost

    active_positions.append({
        "symbol": symbol,
        "buy_date": TODAY,
        "buy_price": float(price),
        "quantity": int(quantity),
        "invested_amount": actual_cost
    })

    blocked_symbols.add(symbol)

    print(
        f"Paper BUY: {quantity} {symbol} at ₹{price:,.2f}; "
        f"cost=₹{actual_cost:,.2f}"
    )


# ------------------------------------------------------------
# 6. Final validation and write paper-account state
# ------------------------------------------------------------

if not np.isfinite(cash_balance) or cash_balance < 0:
    raise ValueError(f"Invalid final paper cash balance: {cash_balance}")

portfolio_records = []

for position in active_positions:
    buy_price = finite_float(position["buy_price"])
    quantity_value = finite_float(position["quantity"])
    invested_amount = finite_float(position.get("invested_amount"))

    if (
        buy_price is None or buy_price <= 0
        or quantity_value is None or quantity_value <= 0
        or quantity_value % 1 != 0
    ):
        raise ValueError(f"Invalid position before writing: {position}")

    quantity = int(quantity_value)

    if invested_amount is None or invested_amount <= 0:
        invested_amount = buy_price * quantity

    portfolio_records.append((
        str(position["symbol"]),
        to_date(position["buy_date"]),
        float(buy_price),
        int(quantity),
        float(invested_amount)
    ))

(
    spark.createDataFrame(portfolio_records, schema=PORTFOLIO_SCHEMA)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_paper_portfolio")
)

cash_records = [("PAPER_TRADER", float(cash_balance), TODAY)]

(
    spark.createDataFrame(cash_records, schema=CASH_SCHEMA)
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_paper_cash")
)

if ledger_rows:
    ledger_records = [
        (
            row["symbol"],
            row["buy_date"],
            row["sell_date"],
            row["buy_price"],
            row["sell_price"],
            row["quantity"],
            row["realized_pnl"],
            row["pnl_pct"],
            row["reason"]
        )
        for row in ledger_rows
    ]

    (
        spark.createDataFrame(ledger_records, schema=LEDGER_SCHEMA)
        .write.format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable("gold_paper_ledger")
    )

print(
    f"\n✅ Paper-trading run complete. "
    f"Open positions: {len(active_positions)} | "
    f"Cash: ₹{cash_balance:,.2f} | "
    f"Closed trades: {len(ledger_rows)}"
)

display(spark.read.table("gold_paper_portfolio"))
display(spark.read.table("gold_paper_cash"))

StatementMeta(, 6b7f7081-f349-439d-a434-efc1c4430c50, 3, Finished, Available, Finished, False)

Run ID: 8131d18a-8905-4a0c-b0c9-f1b2bfa0227e
Paper-trading date: 2026-10-08
Starting simulated cash: ₹276,484.80
Existing simulated positions: 15
Symbols with a recent valid price: 397
Candidate symbols: 15

✅ Paper-trading run complete. Open positions: 15 | Cash: ₹276,484.80 | Closed trades: 0


SynapseWidget(Synapse.DataFrame, 4f2012c7-f92c-457f-85fb-6e813cbf8a7b)

SynapseWidget(Synapse.DataFrame, 6225c94a-8ab4-4a5a-88d1-e8822ccbb424)

In [2]:
display(spark.read.format("csv").option("header","true").load("Files/portfolio/holdings.csv"))

StatementMeta(, 6b7f7081-f349-439d-a434-efc1c4430c50, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3ea00baf-d33c-4e11-8405-5507066a1e72)

In [3]:
df = spark.read.format("csv").option("header","true").load("Files/portfolio/holdings.csv")
# df now is a Spark DataFrame containing CSV data from "Files/portfolio/holdings.csv".
display(df)

StatementMeta(, 6b7f7081-f349-439d-a434-efc1c4430c50, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 132991e3-eeb5-45d8-880b-43dbbdff9a6d)

In [4]:
display(
    spark.sql("""
        SELECT
            portfolio_action,
            COUNT(*) AS total_positions,
            SUM(investment_cost) AS total_cost,
            SUM(current_value) AS total_value,
            SUM(unrealized_pnl) AS total_pnl
        FROM gold_portfolio_actions
        GROUP BY portfolio_action
    """)
)

StatementMeta(, 6b7f7081-f349-439d-a434-efc1c4430c50, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7b55e0e4-8bd8-4070-877a-ed333228ffdf)

In [5]:
display(
    spark.sql("""
        SELECT
            symbol,
            buy_price,
            current_price,
            pnl_pct,
            composite_score,
            eligibility_status,
            portfolio_action,
            action_reason
        FROM gold_portfolio_actions
        ORDER BY
            CASE portfolio_action 
                WHEN 'EXIT_CANDIDATE' THEN 1
                WHEN 'UNDER_REVIEW' THEN 2
                ELSE 3 END,
            pnl_pct ASC
    """)
)

StatementMeta(, 6b7f7081-f349-439d-a434-efc1c4430c50, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 08fc07f7-1727-406c-bdbb-ba2bb80bb2a8)

In [6]:
import pandas as pd

# 1. Make sure folder exists
mssparkutils.fs.mkdirs("Files/portfolio")

# 2. Create header-only file
empty = pd.DataFrame(columns=["symbol","buy_date","buy_price","quantity","broker"])
empty.to_csv("/lakehouse/default/Files/portfolio/holdings.csv", index=False)

print("Recreated empty holdings.csv")
display(spark.read.format("csv").option("header","true").load("Files/portfolio/holdings.csv"))

StatementMeta(, 6b7f7081-f349-439d-a434-efc1c4430c50, 8, Finished, Available, Finished, False)

Recreated empty holdings.csv


SynapseWidget(Synapse.DataFrame, 849e064e-23d8-4a3e-bb74-1e722146a519)